Chart Analysis
Use Python to retrieve stock market data and analyze price changes.
1. Fetch both daily and hourly stock market data using a suitable API.
Visualize the data in several types of plots.
2. Plot long-term stock data over multiple years.
Create an interactive graph that allows the user to view different time ranges.
3. Start with Apple (AAPL) and plot its stock chart using daily data over several years.
4. Add moving averages to the stock charts, such as:
○ SMA20 and SMA50, or
○ SMA50 and SMA150
These should be overlaid on the main stock price chart.
5. Create a table for five stocks. Use default examples such as FAANG stocks plus
NVIDIA, but allow the user to change them.
For each stock, calculate the historical probability that the next day is positive if the
current day changes by:
+2%, +3%, +4%, +5%, and +6%.
Example: if Apple rises 5% today, what is the historical probability that the next day
closes positive?
Use the last 2 years of historical data.
6. Repeat the same analysis for negative daily moves, such as:
-2%, -3%, -4%, and -5%.
7. Repeat both of the above analyses again using 5 years of historical data.
8. Create a comparison table for the five selected stocks.
Include a simple correlation / collinearity analysis between them.
9. Plot a comparison chart for two to five stocks using daily data for one year.
Example: compare Apple and Amazon.
Normalize or rescale prices if needed so they can be compared clearly.
10. Create another table for the same five stocks showing the probability that the next day
is positive after:
● 2 consecutive positive days
● 3 consecutive positive days
● 4 consecutive positive days
● 5 consecutive positive days
● 6 consecutive positive days
11. Repeat the same analysis for consecutive negative days.
12. Screen all Nasdaq and NYSE stocks and identify the top 10 trending stocks, ranked
by daily percentage change.
13. Repeat the screening, but only include stocks with a market capitalization of $5 billion
or more.
This may require obtaining market cap data from an API or web scraping source.

I will be using the Massive API, formerly polygon.io. Massive obtains its data from an aggregate of all major US stock exchanges. The plan I am selecting provides 5 years of historical data.

In [ ]:
import requests
import pandas as pd
from pathlib import Path
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta

# Bypass scientific notation
pd.options.display.float_format = '{:,.0f}'.format


def load_api_key(filepath="api_keys/massive.txt"):
    """Load Massive API key from a local text file."""
    return Path(filepath).read_text(encoding="utf-8").strip()


def get_all_pages(url, params=None):
    """Fetch all paginated results from Massive."""
    all_results = []

    while url:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()

        if data.get("status") == "NOT_AUTHORIZED":
            raise PermissionError(data.get("message", "Not authorized for this request."))

        results = data.get("results", [])
        all_results.extend(results)

        # After the first request, next_url already contains the needed query info
        url = data.get("next_url")
        params = None

    return all_results


def get_massive_daily_bars(
    ticker="AAPL",
    api_key_path="api_keys/massive.txt",
    years=5,
    safety_days=1,
    adjusted=True,
    sort="asc"
):
    """
    Fetch daily aggregate bars for a ticker from Massive.

    Parameters
    ----------
    ticker : str
        Stock ticker symbol, e.g. 'AAPL'
    api_key_path : str
        Path to text file containing API key
    years : int
        Number of years of history to request
    safety_days : int
        Number of days to move forward from exact cutoff to avoid entitlement edge issues
    adjusted : bool
        Whether to return adjusted prices
    sort : str
        'asc' or 'desc'

    Returns
    -------
    pd.DataFrame
    """
    api_key = load_api_key(api_key_path)

    today = date.today()
    start_date = today - relativedelta(years=years) + timedelta(days=safety_days)

    start_str = start_date.isoformat()
    end_str = today.isoformat()

    url = f"https://api.massive.com/v2/aggs/ticker/{ticker}/range/1/day/{start_str}/{end_str}"

    params = {
        "adjusted": str(adjusted).lower(),
        "sort": sort,
        "limit": 50000,
        "apiKey": api_key,
    }

    results = get_all_pages(url, params)

    if not results:
        raise ValueError(f"No data returned for {ticker}.")

    df = pd.DataFrame(results).rename(columns={
        "t": "timestamp",
        "o": "open",
        "h": "high",
        "l": "low",
        "c": "close",
        "v": "volume",
        "vw": "vwap",
        "n": "transactions",
    })

    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms").dt.date
    df["ticker"] = ticker

    preferred_order = [
        "timestamp", "ticker", "open", "high", "low", "close",
        "volume", "vwap", "transactions"
    ]
    df = df[[col for col in preferred_order if col in df.columns]]

    return df

def fetch_and_save_ticker(
    ticker,
    api_key_path="api_keys/massive.txt",
    years=5,
    safety_days=1,
    adjusted=True,
    sort="asc",
    save_folder="raw_daily_data"
):
    df = get_massive_daily_bars(
        ticker=ticker,
        api_key_path=api_key_path,
        years=years,
        safety_days=safety_days,
        adjusted=adjusted,
        sort=sort
    )

    filename = f"{save_folder}/daily_{ticker.lower()}.csv"
    df.to_csv(filename, index=False)

    print(f"{ticker} saved to {filename}")
    return df


tickers = [
    # Big Tech / Growth
    # META/FB must be handled separately due to name change
    "AAPL", "MSFT", "NVDA", "GOOGL", "AMZN", "TSLA", "NFLX",
    
    # Finance
    "JPM", "BAC", "GS", "MS",
    
    # Consumer / Retail
    "WMT", "COST", "HD", "NKE", "SBUX",
    
    # Healthcare / Pharma
    "JNJ", "PFE", "MRK", "UNH",
    
    # Energy
    "XOM", "CVX",
    
    # Industrials / Transportation
    "BA", "CAT", "GE", "UPS",
    
    # ETFs (nice for comparison)
    "SPY", "QQQ", "DIA"
]



# Maybe better structure, don't worry about it for now until I can do more testing
# tickers_by_sector = {
#     "Tech": ["AAPL", "MSFT", "NVDA", "GOOGL", "AMZN", "META", "NFLX],
#     "Finance": ["JPM", "BAC", "GS", "MS"],
#     "Consumer": ["WMT", "COST", "HD", "NKE", "SBUX"],
#     "Healthcare": ["JNJ", "PFE", "MRK", "UNH"],
#     "Energy": ["XOM", "CVX"],
#     "Industrial": ["BA", "CAT", "GE", "UPS"],
#     "ETFs": ["SPY", "QQQ", "DIA"]
# }






data = {}

for t in tickers:
    data[t] = fetch_and_save_ticker(t)


# Special FB/META handling
# FB was Meta's old ticker through 2022-06-08
# META became the ticker starting 2022-06-09

fb_raw = get_massive_daily_bars("FB")
meta_raw = get_massive_daily_bars("META")

fb_raw["timestamp"] = pd.to_datetime(fb_raw["timestamp"])
meta_raw["timestamp"] = pd.to_datetime(meta_raw["timestamp"])

fb_valid = fb_raw[fb_raw["timestamp"] <= "2022-06-08"].copy()
meta_valid = meta_raw[meta_raw["timestamp"] >= "2022-06-09"].copy()

meta_clean = pd.concat([fb_valid, meta_valid], ignore_index=True)

meta_clean["ticker"] = "META"
meta_clean = meta_clean.sort_values("timestamp").reset_index(drop=True)

# Optional: convert timestamp back to date, matching your original format
meta_clean["timestamp"] = meta_clean["timestamp"].dt.date

# Save clean META file
Path("daily_data").mkdir(exist_ok=True)
meta_clean.to_csv("daily_data/daily_meta.csv", index=False)

# Store in dictionary
data["META"] = meta_clean

print("Clean META saved to raw_daily_data/daily_meta.csv")

AAPL saved to raw_daily_data/daily_aapl.csv
MSFT saved to raw_daily_data/daily_msft.csv
NVDA saved to raw_daily_data/daily_nvda.csv
GOOGL saved to raw_daily_data/daily_googl.csv
AMZN saved to raw_daily_data/daily_amzn.csv
TSLA saved to raw_daily_data/daily_tsla.csv
NFLX saved to raw_daily_data/daily_nflx.csv
JPM saved to raw_daily_data/daily_jpm.csv
BAC saved to raw_daily_data/daily_bac.csv
GS saved to raw_daily_data/daily_gs.csv
MS saved to raw_daily_data/daily_ms.csv
WMT saved to raw_daily_data/daily_wmt.csv
COST saved to raw_daily_data/daily_cost.csv
HD saved to raw_daily_data/daily_hd.csv
NKE saved to raw_daily_data/daily_nke.csv
SBUX saved to raw_daily_data/daily_sbux.csv
JNJ saved to raw_daily_data/daily_jnj.csv
PFE saved to raw_daily_data/daily_pfe.csv
MRK saved to raw_daily_data/daily_mrk.csv
UNH saved to raw_daily_data/daily_unh.csv
XOM saved to raw_daily_data/daily_xom.csv
CVX saved to raw_daily_data/daily_cvx.csv
BA saved to raw_daily_data/daily_ba.csv
CAT saved to raw_daily

In [16]:
ticker_names = {
    # Big Tech / Growth
    "AAPL": "Apple Inc.",
    "MSFT": "Microsoft Corporation",
    "NVDA": "NVIDIA Corporation",
    "GOOGL": "Alphabet Inc. (Class A)",
    "AMZN": "Amazon.com, Inc.",
    "META": "Meta Platforms, Inc.",
    "TSLA": "Tesla, Inc.",
    "NFLX": "Netflix, Inc.",
    
    # Finance
    "JPM": "JPMorgan Chase & Co.",
    "BAC": "Bank of America Corporation",
    "GS": "Goldman Sachs Group, Inc.",
    "MS": "Morgan Stanley",
    
    # Consumer / Retail
    "WMT": "Walmart Inc.",
    "COST": "Costco Wholesale Corporation",
    "HD": "The Home Depot, Inc.",
    "NKE": "NIKE, Inc.",
    "SBUX": "Starbucks Corporation",
    
    # Healthcare / Pharma
    "JNJ": "Johnson & Johnson",
    "PFE": "Pfizer Inc.",
    "MRK": "Merck & Co., Inc.",
    "UNH": "UnitedHealth Group Incorporated",
    
    # Energy
    "XOM": "Exxon Mobil Corporation",
    "CVX": "Chevron Corporation",
    
    # Industrials / Transportation
    "BA": "The Boeing Company",
    "CAT": "Caterpillar Inc.",
    "GE": "General Electric Company",
    "UPS": "United Parcel Service, Inc.",
    
    # ETFs
    "SPY": "SPDR S&P 500 ETF Trust",
    "QQQ": "Invesco QQQ Trust",
    "DIA": "SPDR Dow Jones Industrial Average ETF Trust"
}

In [2]:
from pathlib import Path
import pandas as pd

def update_daily_master(
    ticker,
    api_key_path="api_keys/massive.txt",
    years=5,
    safety_days=1,
    adjusted=True,
    sort="asc",
    save_folder="datasets"
):
    save_path = Path(save_folder)
    save_path.mkdir(parents=True, exist_ok=True)

    master_file = save_path / f"{ticker.lower()}_daily_master.csv"

    new_df = get_massive_daily_bars(
        ticker=ticker,
        api_key_path=api_key_path,
        years=years,
        safety_days=safety_days,
        adjusted=adjusted,
        sort=sort
    )

    if master_file.exists():
        old_df = pd.read_csv(master_file)
        old_df["timestamp"] = pd.to_datetime(old_df["timestamp"]).dt.date

        combined = pd.concat([old_df, new_df], ignore_index=True)
    else:
        combined = new_df.copy()

    combined["timestamp"] = pd.to_datetime(combined["timestamp"]).dt.date

    combined = (
        combined
        .drop_duplicates(subset=["timestamp"], keep="last")
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    combined.to_csv(master_file, index=False)

    print(f"{ticker} master updated: {master_file}")
    print(combined.head())
    print(combined.tail())
    print(combined.shape)
    print("Date range:", combined["timestamp"].min(), "to", combined["timestamp"].max())

    return combined

In [3]:
for ticker in tickers:
    data[ticker] = update_daily_master(ticker)

AAPL master updated: datasets/aapl_daily_master.csv
    timestamp ticker  open  high  low  close      volume  vwap  transactions
0  2021-04-22   AAPL   133   134  131    132  84,566,456   133        615670
1  2021-04-23   AAPL   132   135  132    134  78,666,779   134        519533
2  2021-04-26   AAPL   135   135  134    135  66,888,509   135        484069
3  2021-04-27   AAPL   135   135  134    134  66,015,804   135        480003
4  2021-04-28   AAPL   134   135  133    134 107,746,597   135        783355
       timestamp ticker  open  high  low  close     volume  vwap  transactions
1250  2026-04-15   AAPL   258   267  258    266 49,913,511   264        728560
1251  2026-04-16   AAPL   267   267  261    263 43,323,112   263        635080
1252  2026-04-17   AAPL   267   272  267    270 61,436,228   270        723488
1253  2026-04-20   AAPL   270   274  270    273 36,582,599   273        541032
1254  2026-04-21   AAPL   272   273  265    266 50,192,036   268        710075
(1255, 9)
Da

In [ ]:
aapl_daily = update_daily_master("AAPL")
msft_daily = update_daily_master("MSFT")
nvda_daily = update_daily_master("NVDA")

# Combine all data into one

In [19]:
import pandas as pd
import glob
import os

# 1. Define the directory path
path = 'daily_data' 

# 2. Find all CSV files in that folder
all_files = glob.glob(os.path.join(path, "*.csv"))

# 3. Read each file and store in a list
df_list = []
for filename in all_files:
    df = pd.read_csv(filename)
    # Optional: Add a column to track which file the data came from
    df['source_file'] = os.path.basename(filename)
    df_list.append(df)

# 4. Combine everything into one master DataFrame
combined_df = pd.concat(df_list, ignore_index=True)

print(combined_df.head())

    timestamp ticker  open  high  low  close      volume  vwap  transactions  \
0  2021-04-28   AAPL   134   135  133    134 107,746,597   135        783355   
1  2021-04-29   AAPL   136   137  132    133 151,100,953   134       1059387   
2  2021-04-30   AAPL   132   134  131    131 109,425,466   132        701292   
3  2021-05-03   AAPL   132   134  132    133  75,135,100   133        580631   
4  2021-05-04   AAPL   131   131  127    128 137,544,918   128       1182700   

      source_file  
0  aapl_daily.csv  
1  aapl_daily.csv  
2  aapl_daily.csv  
3  aapl_daily.csv  
4  aapl_daily.csv  


In [20]:
combined_df

,timestamp,ticker,open,high,low,close,volume,vwap,transactions,source_file
0,2021-04-28,AAPL,134,135,133,134,"107,746,597",135,783355,aapl_daily.csv
1,2021-04-29,AAPL,136,137,132,133,"151,100,953",134,1059387,aapl_daily.csv
2,2021-04-30,AAPL,132,134,131,131,"109,425,466",132,701292,aapl_daily.csv
3,2021-05-03,AAPL,132,134,132,133,"75,135,100",133,580631,aapl_daily.csv
4,2021-05-04,AAPL,131,131,127,128,"137,544,918",128,1182700,aapl_daily.csv
...,...,...,...,...,...,...,...,...,...,...
37645,2026-04-21,MRK,116,117,112,113,"12,766,661",113,151882,mrk_daily.csv
37646,2026-04-22,MRK,113,114,112,113,"8,037,515",113,101114,mrk_daily.csv
37647,2026-04-23,MRK,113,115,113,115,"6,826,250",114,93151,mrk_daily.csv
37648,2026-04-24,MRK,114,114,112,112,"7,060,016",112,93014,mrk_daily.csv


In [21]:
combined_df['ticker'].value_counts()

ticker
AAPL     1255
JNJ      1255
QQQ      1255
COST     1255
MSFT     1255
AMZN     1255
TSLA     1255
CVX      1255
DIA      1255
HD       1255
UNH      1255
PFE      1255
GE       1255
CAT      1255
GOOGL    1255
JPM      1255
SBUX     1255
NFLX     1255
SPY      1255
META     1255
UPS      1255
XOM      1255
BAC      1255
NVDA     1255
GS       1255
NKE      1255
MS       1255
WMT      1255
BA       1255
MRK      1255
Name: count, dtype: int64

In [18]:
# Why is this rounding? Wasn't doing that before
df_daily_meta = pd.read_csv('daily_data/meta_daily.csv')
df_daily_meta

,timestamp,ticker,open,high,low,close,volume,vwap,transactions
0,2021-04-28,META,307,311,305,307,"33,907,210",311,398610
1,2021-04-29,META,330,332,322,330,"56,426,771",327,597569
2,2021-04-30,META,326,330,324,325,"26,282,423",326,272188
3,2021-05-03,META,326,329,322,323,"18,719,462",324,248771
4,2021-05-04,META,320,322,313,318,"24,032,557",316,291992
...,...,...,...,...,...,...,...,...,...
1250,2026-04-21,META,671,676,667,669,"8,655,071",671,271151
1251,2026-04-22,META,674,678,670,675,"9,215,604",675,276840
1252,2026-04-23,META,664,670,653,659,"11,666,981",661,362722
1253,2026-04-24,META,660,681,654,675,"13,349,635",672,367938
